# Geometric GO

> Geometric genomic offset

The geometric genomic offset. 

TODO!

In [ ]:
#| default_exp geometric

In [ ]:
#| export
from genomic_offsets.RidgeLFMM import *
from fastcore.utils import *
import statsmodels.api as sm
from nbdev.showdoc import *
from numba import njit, jit
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [ ]:
#| export
class GeometricGO:
    "Geometric genomic offset statistic."
    def __init__(self, 
                 K: int, # Number of latent factors 
                 lambd: float): # Regularization parameter
        self.K = K
        self.lambd = lambd
        self._mx = None
        self._sx = None
        self.Cb = None
    def __str__(self):
        return f"Geometric genomic offset with K={self.K} and lambda={self.lambd}"
    __repr__ = __str__

In order to use the model we have first to initialize it with the number of latent factors $K$ and a certain regularization parameter $\lambda$. 

In [ ]:
model = GeometricGO(K=1, lambd=1e-5)

Then, we have to fit the model ...

In [ ]:
#|export
@patch
def fit(self:GeometricGO,
        Y: np.ndarray, # Allele frequency matrix (nxL)
        X: np.ndarray): # Environmental matrix (nxP)
    "Fits the Geometric genomic offset model. "
    n1, L = Y.shape
    n2, P = X.shape
    if n1 != n2: 
        raise ValueError("Dimensions of array don't match")
    # Scale data and save it to predict later
    Y = Y - np.mean(Y, axis=0)
    mx = np.mean(X, axis=0)
    X = X - mx
    sx = np.std(X, axis=0)
    X = X / sx
    self._mx = mx
    self._sx = sx
    model = RidgeLFMM(K=self.K, lambd=self.lambd)
    model.fit(Y=Y, X=X)
    self.Cb = np.dot(model.B.T, model.B) / model.B.shape[0]

The `fit()` method expects as input an allele matrix $\mathbf Y$ and an environmental matrix $\mathbf X$ with as many rows as individuals. For now, let us simulate them under the ¿generative model?: 

In [ ]:
def generative_model(rng, N, L, P, n_targets):
    X = rng.normal(size=N)
    B = np.zeros(L)
    target_indices = rng.choice(L, n_targets, replace=False)
    B[target_indices] = rng.uniform(-10, 10, size=n_targets)
    U = np.dot(X.reshape(-1, 1), np.array([[-1, 0.5, 1.5]])) + rng.normal(size=(N, 3))
    V = rng.normal(size=(3, L))  # V should have 3 rows to match the columns of U
    Y = np.dot(X.reshape(-1, 1), B.reshape(1, -1)) + np.dot(U, V) + rng.normal(scale=0.5, size=(N, L))
    Y = (Y > 0).astype(int)
    X = np.hstack((X.reshape(-1, 1),  rng.normal(size=(N, P-1))))
    assert X.shape == (N, P)
    assert Y.shape == (N, L)
    return Y, X
rng = np.random.default_rng(1000)
Y, X = generative_model(rng, N=100, L=500, P=10, n_targets=10)

In [ ]:
indices = rng.permutation(100)
training_idx, test_idx = indices[:60], indices[60:]
Xtrain, Xpredict = X[training_idx,:], X[test_idx,:]
Ytrain, Ypredict = Y[training_idx,:], Y[test_idx,:]
model.fit(Ytrain, Xtrain)

Finally, we can predict the genomic offset under two different environments: 

In [ ]:
#| hide
#| export
@patch
def _rescale_env(self:GeometricGO,
        X: np.ndarray, # Environmental matrix (nxP)
        )-> np.ndarray: # A vector of genomic offsets (n)
    if self._mx is None or self._sx  is None: 
        raise ValueError("You have to fit the model first!")
    return (X-self._mx) / self._sx 

@njit
def genetic_gap(Cb: np.ndarray, X: np.ndarray, Xstar: np.ndarray) -> np.ndarray:
    offsets = np.zeros(X.shape[0])
    for i in range(len(offsets)):
        diff = X[i, :] - Xstar[i, :]
        offsets[i] = np.dot(np.dot(diff, Cb), diff.T)
    return offsets

In [ ]:
#| export 
@patch
def genomic_offset(self:GeometricGO,
        X: np.ndarray, # Environmental matrix (nxP)
        Xstar: np.ndarray, # Altered environmental matrix (nxP)
           )-> np.ndarray: # A vector of genomic offsets (n)
    "Calculates the genomic offset statistic. " 
    if X.shape != Xstar.shape: 
        raise ValueError("Dimensions of array don't match")
    return genetic_gap(self.Cb, self._rescale_env(X), self._rescale_env(Xstar))

As expected, the genomic offset is zero if both environmental matrixes are identical: 

In [ ]:
model.genomic_offset(Xpredict, Xpredict)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0.])

In [ ]:
offset = model.genomic_offset(Xpredict, Xpredict+rng.normal(size=Xpredict.shape))
offset

array([0.45521477, 0.09006635, 0.17185525, 0.11570215, 0.00896073,
       0.04192136, 0.0292939 , 0.26143774, 0.24471996, 0.0064966 ,
       0.22319557, 0.04904645, 0.17064012, 0.09292498, 0.23717266,
       0.120408  , 0.02731037, 0.08126873, 0.04341562, 0.06502931,
       0.11710134, 0.11759253, 0.02034091, 0.03001484, 0.00405735,
       0.01403853, 0.03992977, 0.05160579, 0.06869659, 0.36102953,
       0.11020969, 0.01519932, 0.01971894, 0.03587198, 0.02604959,
       0.06261495, 0.0313995 , 0.06330403, 0.01603789, 0.02613388])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()